In [1]:
import pandas as pd

In [2]:
#import data
players = pd.read_csv("data/players.csv", parse_dates=["signup_date"])
matches = pd.read_csv("data/matches.csv", parse_dates=["match_date"])

In [3]:
#merge
df = matches.merge(players, on="player_id")

In [4]:
#calculate days since signup
df["days_since_signup"] = (df["match_date"] - df["signup_date"]).dt.days

In [5]:
#summary
summary = df.groupby("player_id").agg(
    total_matches=("win","count"),
    win_rate=("win","mean"),
    avg_skill_diff=("player_skill","mean")
).reset_index()

In [6]:
#designate early behaviour
early = df[df["days_since_signup"] <= 3]

early_features = early.groupby("player_id").agg(
    early_matches=("win","count"),
    early_win_rate=("win","mean")
).reset_index()

In [7]:
#churn
last_match = df.groupby("player_id")["match_date"].max().reset_index()
max_date = df["match_date"].max()

last_match["churn"] = ((max_date - last_match["match_date"]).dt.days > 7).astype(int)

In [8]:
#final dataset
final = summary.merge(early_features, on="player_id", how="left")\
               .merge(last_match[["player_id","churn"]], on="player_id")

final.fillna(0, inplace=True)

final.to_csv("data/player_features.csv", index=False)